# Tahap 2 — Standardisasi dan Harmonisasi Data

**Sumber kebenaran:** `implementation_playbook_dataset_lstm.md`, bagian Tahap 2.

**Ruang lingkup notebook ini (dan hanya ini):**
- Standardisasi format tanggal (Ogimet & SounderPy)
- Standardisasi tipe data numerik
- Standardisasi placeholder Ogimet (`Tr`, `----`, `-----`)
- Identifikasi & dokumentasi (dan parsing HANYA jika terbukti konsisten) format numerik abnormal SounderPy
- Harmonisasi nama kolom ke satu konvensi tunggal

**Tidak dilakukan di notebook ini** (di luar scope Tahap 2 — lihat sel dokumentasi di akhir):
drop duplicate, drop missing value, imputasi, interpolasi, merge dataset, seleksi sounding 12Z/00Z,
master calendar, labeling, feature engineering.

**Konteks tervalidasi dari Tahap 1 (tidak diaudit ulang di sini):**
- Ogimet: delimiter `;`, encoding `latin-1`, kolom tanggal `Date`; placeholder `Tr`→`0.0`, `----`→`NaN`, `-----`→`NaN`; 365 duplicate rows (DIPERTAHANKAN).
- SounderPy: delimiter `;`, encoding `latin-1`, kolom tanggal `nominal_date`, kolom jam `observation_hour`; beberapa kolom numerik masih object karena format seperti `2.226.401`.

**Revisi yang disetujui sebelum notebook ini dibuat:**
1. CAPE ikut distandardisasi dan disertakan di output (bukan fitur model, dipakai untuk labeling di Tahap 8).
2. Format numerik abnormal SounderPy **tidak** diubah berdasarkan asumsi — hanya diidentifikasi, didokumentasikan, dan diparsing jika aturan formatnya terbukti konsisten secara data-driven.
3. Nama kolom final: `date`, `hour`, `rr`, `tavg`, `rh`, `cape`, `cin`, `kindex`, `li`, `tt`, `sweat`.
4. Duplicate Ogimet tetap dipertahankan (tidak di-drop).
5. Isu UTC vs WIB pada jam sounding hanya dicatat sebagai risiko, tidak diselesaikan di sini.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import glob
import os

pd.set_option("display.max_columns", None)


### 0.1 Path data mentah

Sesuaikan path berikut dengan lokasi file mentah di Google Drive / Colab kamu.
Jika data tersebar di banyak file (misal per tahun), gunakan `glob` untuk mengumpulkan seluruh path,
lalu load & concat — TANPA menghapus duplicate dan TANPA menggabungkan kedua sumber (itu Tahap 3/5).


In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

OGIMET_RAW_PATHS = ["ogimet_dataset.csv"]
SOUNDERPY_RAW_PATHS = ["indeks_atmosfer_dataset.csv"]

OUTPUT_DIR = "02_pendukung"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Jumlah file Ogimet mentah   : {len(OGIMET_RAW_PATHS)}")
print(f"Jumlah file SounderPy mentah: {len(SOUNDERPY_RAW_PATHS)}")


Jumlah file Ogimet mentah   : 1
Jumlah file SounderPy mentah: 1


## 1. Ogimet — Load Data Mentah

Load seluruh file mentah Ogimet (delimiter `;`, encoding `latin-1`) dan gabungkan menjadi satu
DataFrame sementara. Ini BUKAN penggabungan dengan SounderPy — hanya penggabungan antar file
dalam satu sumber yang sama, sesuai Tahap 1.


In [3]:
def load_ogimet_raw(paths):
    frames = []
    for p in paths:
        df = pd.read_csv(p, delimiter=";", encoding="latin-1")
        frames.append(df)
    if not frames:
        raise FileNotFoundError("Tidak ada file Ogimet mentah ditemukan. Cek OGIMET_RAW_PATHS.")
    return pd.concat(frames, ignore_index=True)

ogimet_raw = load_ogimet_raw(OGIMET_RAW_PATHS)
print(ogimet_raw.shape)
ogimet_raw.head()


(3287, 7)


,Date,Daily Rainfall (mm),Average Air Temperature (°C),Relative Humidity (%),Pressure,Wind Speed,Wind Direction
0,01/01/2017,----,28.2,83.9,1008.9,4.6,WSW
1,02/01/2017,26.9,26.5,87.4,1009.8,7.2,W
2,03/01/2017,78.0,26.2,87.6,1010.3,27.6,NNE
3,04/01/2017,33.0,24.7,92.4,1010.1,5.3,WSW
4,05/01/2017,83.0,25.7,88.3,1009.7,3.5,NW


## 2. Ogimet — Standardisasi Tanggal

Parse kolom `Date` ke format tunggal `YYYY-MM-DD`. Baris yang gagal di-parse (NaT) dilaporkan
secara eksplisit (bukan dibuang) — sesuai validasi Tahap 2: "semua kolom tanggal berhasil
di-parse tanpa error/NaT". Jika ada NaT, ini adalah temuan yang harus dicatat, bukan diam-diam
diabaikan.


In [ ]:
ogimet_raw["date"] = pd.to_datetime(
    ogimet_raw["Date"],
    dayfirst=True,
    errors="coerce"
)
n_nat = ogimet_raw["date"].isna().sum()
print(f"Baris Ogimet dengan tanggal gagal di-parse (NaT): {n_nat}")
if n_nat > 0:
    display(ogimet_raw[ogimet_raw["date"].isna()])


## 3. Ogimet — Standardisasi Placeholder

Keputusan metodologi (sudah difiksasi, Tahap 1):
- `Tr` → `0.0` (trace rainfall, dianggap curah hujan sangat kecil/nol)
- `----` → `NaN`
- `-----` → `NaN`

Placeholder diganti pada SELURUH kolom variabel penelitian sebelum konversi tipe data,
supaya konversi ke numerik tidak gagal karena string placeholder.


In [5]:
PLACEHOLDER_MAP = {
    "Tr": 0.0,
    "----": np.nan,
    "-----": np.nan,
}

RAW_VAR_COLS_OGIMET = [
    "Daily Rainfall (mm)",
    "Average Air Temperature (°C)",
    "Relative Humidity (%)"
]
for col in RAW_VAR_COLS_OGIMET:
    before_placeholder_counts = ogimet_raw[col].astype(str).value_counts().reindex(PLACEHOLDER_MAP.keys(), fill_value=0)
    print(f"Kolom '{col}' — jumlah placeholder ditemukan:")
    print(before_placeholder_counts)
    print("-" * 40)
    ogimet_raw[col] = ogimet_raw[col].replace(PLACEHOLDER_MAP)


Kolom 'Daily Rainfall (mm)' — jumlah placeholder ditemukan:
Daily Rainfall (mm)
Tr       184
----      37
-----      0
Name: count, dtype: int64
----------------------------------------
Kolom 'Average Air Temperature (°C)' — jumlah placeholder ditemukan:
Average Air Temperature (°C)
Tr        0
----      0
-----    31
Name: count, dtype: int64
----------------------------------------
Kolom 'Relative Humidity (%)' — jumlah placeholder ditemukan:
Relative Humidity (%)
Tr        0
----      0
-----    17
Name: count, dtype: int64
----------------------------------------


## 4. Ogimet — Standardisasi Tipe Data Numerik

Pastikan `RR`, `Tavg`, `RH` benar-benar numerik (float), bukan string berisi angka.
`errors="coerce"` di sini hanya untuk menangkap sisa nilai non-numerik yang TIDAK termasuk
tiga placeholder yang sudah didefinisikan — jika muncul, ini temuan baru yang harus dilaporkan,
bukan diam-diam di-NaN-kan tanpa catatan.


In [6]:
unexpected_nonnumeric = {}

for col in RAW_VAR_COLS_OGIMET:
    numeric_col = pd.to_numeric(ogimet_raw[col], errors="coerce")
    # nilai yang jadi NaN padahal aslinya BUKAN salah satu placeholder yang sudah didefinisikan
    was_placeholder_or_nan = ogimet_raw[col].isna()
    newly_nan = numeric_col.isna() & ~was_placeholder_or_nan
    if newly_nan.any():
        unexpected_nonnumeric[col] = ogimet_raw.loc[newly_nan, col].unique().tolist()
    ogimet_raw[col] = numeric_col

print("Nilai non-numerik tak terduga (di luar placeholder yang sudah didefinisikan):")
print(unexpected_nonnumeric if unexpected_nonnumeric else "Tidak ada — semua nilai tertangani.")


Nilai non-numerik tak terduga (di luar placeholder yang sudah didefinisikan):
Tidak ada — semua nilai tertangani.


## 5. Ogimet — Harmonisasi Nama Kolom

Rename ke konvensi final: `date`, `rr`, `tavg`, `rh`.


In [ ]:
ogimet_standardized = ogimet_raw.rename(columns={
    "Daily Rainfall (mm)": "rr",
    "Average Air Temperature (°C)": "tavg",
    "Relative Humidity (%)": "rh",
})[["date", "rr", "tavg", "rh"]].copy()

for col in ["rr", "tavg", "rh"]:
    assert pd.api.types.is_numeric_dtype(ogimet_standardized[col]), f"Kolom {col} belum numerik!"

assert pd.api.types.is_datetime64_any_dtype(ogimet_standardized["date"]), (
    "Kolom date harus tetap datetime64 di sepanjang notebook; konversi ke string "
    "hanya boleh dilakukan saat ekspor final."
)

print("Tipe data akhir:")
print(ogimet_standardized.dtypes)
ogimet_standardized.head()


## 6. Ogimet — Verifikasi Duplicate TIDAK Terhapus

Sesuai instruksi eksplisit: 365 duplicate rows yang ditemukan di Tahap 1 **tidak boleh dihapus**
di Tahap 2. Sel ini hanya memverifikasi jumlah baris konsisten dengan hasil audit — bukan
melakukan tindakan apa pun terhadap duplikat.


In [ ]:
n_rows_before = len(ogimet_raw)
n_rows_after = len(ogimet_standardized)
n_duplicate_rows = ogimet_standardized.duplicated().sum()

print(f"Jumlah baris sebelum standardisasi : {n_rows_before}")
print(f"Jumlah baris setelah standardisasi  : {n_rows_after}")
print(f"Jumlah duplicate rows (dipertahankan): {n_duplicate_rows}")
print(f"Tanggal minimum                     : {ogimet_standardized['date'].min()}")
print(f"Tanggal maksimum                     : {ogimet_standardized['date'].max()}")

assert n_rows_before == n_rows_after, (
    "Jumlah baris berubah selama standardisasi — seharusnya tidak ada baris yang "
    "ditambah/dihapus di Tahap 2."
)


## 7. Ogimet — Simpan Output

In [ ]:
ogimet_export = ogimet_standardized.copy()
ogimet_export["date"] = ogimet_export["date"].dt.strftime("%Y-%m-%d")

ogimet_out_path = os.path.join(OUTPUT_DIR, "ogimet_standardized.csv")
ogimet_export.to_csv(ogimet_out_path, index=False)
print(f"Tersimpan: {ogimet_out_path}")


---
## 8. SounderPy — Load Data Mentah

Sama seperti Ogimet: load seluruh file mentah SounderPy (delimiter `;`, encoding `latin-1`),
gabungkan antar file SounderPy saja (bukan dengan Ogimet).


In [10]:
def load_sounderpy_raw(paths):
    frames = []
    for p in paths:
        df = pd.read_csv(p, delimiter=";", encoding="latin-1")
        frames.append(df)
    if not frames:
        raise FileNotFoundError("Tidak ada file SounderPy mentah ditemukan. Cek SOUNDERPY_RAW_PATHS.")
    return pd.concat(frames, ignore_index=True)

sounderpy_raw = load_sounderpy_raw(SOUNDERPY_RAW_PATHS)
print(sounderpy_raw.shape)
sounderpy_raw.head()


(2774, 29)


,observation_datetime,nominal_date,observation_hour,status,SBCAPE,SBCIN,MLCAPE,MLCIN,MUCAPE,MUCIN,DCAPE,MU_ECAPE,ML_ECAPE,SB_ECAPE,SFC_PRESSURE_hPa,SRH_0_1km,SRH_0_3km,SHEAR_0_6km_kt,EHI_0_3km,SCP,STP,LI_SB_500,TT,KINDEX,SWEAT,n_indices_computed,n_sharppy_direct_computed,reason,sharppy_direct_reason
0,01/01/2017 11:30,01/01/2017,12,SUCCESS,2.226.401,-4.906,1108.96,-7.958,2.226.401,-4.906,1.019.854,NaN,335.147,NaN,1.008.343,4.45,-22.667,8.13,-0.315,0.057,0.000,-4.510,42.2,34.4,219.162,136,4,NaN,NaN
1,02/01/2017 11:30,02/01/2017,12,SUCCESS,921.164,-22.887,195.89,-27.464,921.164,-22.887,462.183,NaN,138.538,NaN,1.008.344,-65.538,-70.233,7.656,-0.404,-0.000,0.000,-2.955,41.5,35.2,223.381,136,4,NaN,NaN
2,03/01/2017 11:30,03/01/2017,12,SUCCESS,1.671.572,0.000,38.018,-33.520,3.440.562,0.000,-0.0,NaN,35.023,NaN,1.008.343,-12.405,84.944,9.97,1.827,0.000,0.000,-3.317,40.5,36.4,277.368,136,4,NaN,NaN
3,04/01/2017 11:30,04/01/2017,12,SUCCESS,397.409,-4.865,217.717,-2.968,397.409,-4.865,301.148,63.084,176.109,63.084,1.008.347,68.198,72.303,31.028,0.180,0.507,0.133,-0.752,39.0,35.1,294.56,136,4,NaN,NaN
4,05/01/2017 11:30,05/01/2017,12,SUCCESS,2.064.721,-15.811,1.636.534,-8.663,2.064.721,-15.811,-0.0,NaN,NaN,NaN,1.008.345,31.139,68.328,46.21,0.882,2.832,0.944,-5.124,44.1,38.0,299.131,136,4,NaN,NaN


## 9. SounderPy — Standardisasi Tanggal & Jam

`nominal_date` → `date` (format `YYYY-MM-DD`).
`observation_hour` → `hour`, distandardisasi bentuknya saja (mis. dipastikan berupa string
`"00Z"` / `"12Z"` yang konsisten). **Seleksi 12Z→00Z fallback TIDAK dilakukan di sini** — itu
Tahap 4. Isu potensi pergeseran tanggal akibat UTC vs waktu lokal (WIB) DICATAT sebagai risiko,
tidak diselesaikan di notebook ini.


In [ ]:
sounderpy_raw["date"] = pd.to_datetime(
    sounderpy_raw["nominal_date"],
    dayfirst=True,
    errors="coerce"
)
n_nat_sp = sounderpy_raw["date"].isna().sum()
print(f"Baris SounderPy dengan tanggal gagal di-parse (NaT): {n_nat_sp}")
if n_nat_sp > 0:
    display(sounderpy_raw[sounderpy_raw["date"].isna()])

sounderpy_raw["hour"] = sounderpy_raw["observation_hour"].astype(str).str.strip().str.upper()
print("Nilai unik kolom hour setelah standardisasi bentuk:")
print(sounderpy_raw["hour"].value_counts())

print("\n[CATATAN RISIKO - TIDAK DISELESAIKAN DI TAHAP 2]")
print("Interpretasi 00Z/12Z terhadap tanggal kalender lokal (WIB, UTC+7) belum diverifikasi.")
print("Berpotensi menggeser tanggal kalender jika tidak konsisten. Ditangani di Tahap 3/4/5.")


## 10. SounderPy — Identifikasi & Dokumentasi Pola Format Numerik Abnormal

Ditemukan nilai seperti `2.226.401` dan `1.008.343` pada beberapa kolom numerik SounderPy,
menyebabkan kolom tersebut masih bertipe object/string.

**Aturan yang disepakati:** Tahap 2 hanya boleh (a) mengidentifikasi pola, (b) mendokumentasikan
pola, dan (c) melakukan parsing HANYA jika aturan format dapat dibuktikan konsisten secara
data-driven — bukan berdasarkan asumsi/tebakan.

**Pembuktian yang digunakan di sini (bukan asumsi):**
Untuk setiap nilai bertitik-ganda seperti `2.226.401`, pisahkan berdasarkan `.` menjadi
beberapa grup digit. Jika format tersebut adalah pemisah ribuan yang konsisten, maka SELURUH
grup SETELAH grup pertama harus berukuran tepat 3 digit (mis. `2.226.401` → grup `226` dan `401`,
keduanya 3 digit). Aturan ini diuji ke SELURUH nilai bertitik-ganda pada kolom tersebut:

- Jika 100% nilai memenuhi aturan "semua grup setelah grup pertama = 3 digit" → pola terbukti
  konsisten dengan notasi pemisah ribuan integer, dan parsing (hapus semua titik → cast ke
  numerik) boleh dilakukan.
- Jika ada SATU SAJA nilai yang melanggar aturan tersebut (grup terakhir bukan 3 digit,
  mengindikasikan kemungkinan itu adalah desimal, bukan ribuan) → pola dianggap TIDAK terbukti
  konsisten. Kolom TIDAK diparsing, tetap disimpan sebagai string asli, dan didokumentasikan
  sebagai temuan terbuka untuk ditinjau peneliti.

Ini bukan interpretasi numerik spekulatif — ini adalah uji struktural terhadap bentuk string,
dan hasilnya (lolos/gagal) dilaporkan apa adanya.


In [12]:
print("Ogimet NaT:", ogimet_raw["date"].isna().sum())
print("SounderPy NaT:", sounderpy_raw["date"].isna().sum())

Ogimet NaT: 365
SounderPy NaT: 0


In [13]:
RAW_VAR_COLS_SOUNDERPY = {
    "SBCIN": "cin",
    "KINDEX": "kindex",
    "LI_SB_500": "li",
    "TT": "tt",
    "SWEAT": "sweat",
    "SBCAPE": "cape",   # nama kolom mentah CAPE - sesuaikan jika berbeda di file asli
}

MULTI_DOT_PATTERN = re.compile(r"^\s*-?\d+(\.\d+){2,}\s*$")  # minimal 2 titik -> berpotensi ambigu

def analyze_multi_dot_pattern(series: pd.Series):
    """Uji apakah SELURUH nilai bertitik-ganda pada suatu kolom konsisten dengan
    notasi pemisah-ribuan (semua grup setelah grup pertama = 3 digit).
    Mengembalikan dict ringkasan bukti, bukan keputusan yang diasumsikan."""
    s = series.astype(str).str.strip()
    matches = s[s.str.match(MULTI_DOT_PATTERN, na=False)]

    total_values = len(s)
    n_multi_dot = len(matches)

    if n_multi_dot == 0:
        return {
            "n_multi_dot_values": 0,
            "n_total_values": total_values,
            "consistent_thousand_sep": None,
            "n_violations": 0,
            "example_violations": [],
            "example_matches": [],
        }

    def groups_after_first_are_3digit(val: str) -> bool:
        v = val.lstrip("-")
        parts = v.split(".")
        return all(len(p) == 3 for p in parts[1:])

    violation_mask = ~matches.apply(groups_after_first_are_3digit)
    n_violations = violation_mask.sum()

    return {
        "n_multi_dot_values": n_multi_dot,
        "n_total_values": total_values,
        "consistent_thousand_sep": bool(n_violations == 0),
        "n_violations": int(n_violations),
        "example_violations": matches[violation_mask].unique().tolist()[:5],
        "example_matches": matches.unique().tolist()[:5],
    }

numeric_format_report = {}
for raw_col in RAW_VAR_COLS_SOUNDERPY:
    if raw_col not in sounderpy_raw.columns:
        print(f"[LEWATI] Kolom mentah '{raw_col}' tidak ditemukan di file SounderPy.")
        continue
    report = analyze_multi_dot_pattern(sounderpy_raw[raw_col])
    numeric_format_report[raw_col] = report
    print(f"Kolom '{raw_col}':")
    for k, v in report.items():
        print(f"  {k}: {v}")
    print("-" * 50)


Kolom 'SBCIN':
  n_multi_dot_values: 0
  n_total_values: 2774
  consistent_thousand_sep: None
  n_violations: 0
  example_violations: []
  example_matches: []
--------------------------------------------------
Kolom 'KINDEX':
  n_multi_dot_values: 0
  n_total_values: 2774
  consistent_thousand_sep: None
  n_violations: 0
  example_violations: []
  example_matches: []
--------------------------------------------------
Kolom 'LI_SB_500':
  n_multi_dot_values: 0
  n_total_values: 2774
  consistent_thousand_sep: None
  n_violations: 0
  example_violations: []
  example_matches: []
--------------------------------------------------
Kolom 'TT':
  n_multi_dot_values: 0
  n_total_values: 2774
  consistent_thousand_sep: None
  n_violations: 0
  example_violations: []
  example_matches: []
--------------------------------------------------
Kolom 'SWEAT':
  n_multi_dot_values: 1
  n_total_values: 2774
  consistent_thousand_sep: True
  n_violations: 0
  example_violations: []
  example_matches: ['

### 10.1 Keputusan Parsing per Kolom (Berdasarkan Bukti, Bukan Asumsi)

Kolom dengan `consistent_thousand_sep = True` DAN `n_multi_dot_values > 0` → diparsing
(hapus semua titik, cast ke numerik).
Kolom lain (tidak ada nilai bertitik-ganda, atau ada pelanggaran) → TIDAK diparsing di sini;
tetap disimpan sebagai string asli dan dicatat sebagai temuan terbuka.


In [ ]:
def parse_thousand_separated(val):
    """Hapus pemisah ribuan dan cast ke float. Hanya dipanggil setelah pola
    terbukti konsisten (lihat pengecekan consistent_thousand_sep di cell audit)."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if MULTI_DOT_PATTERN.match(s):
        sign = -1 if s.startswith("-") else 1
        s_clean = s.lstrip("-")
        parts = s_clean.split(".")
        joined = parts[0] + "".join(parts[1:])
        return sign * float(joined)
    return pd.to_numeric(s, errors="coerce")

columns_parsed = []
columns_flagged_unresolved = []

for raw_col, final_name in RAW_VAR_COLS_SOUNDERPY.items():

    if raw_col not in sounderpy_raw.columns:
        continue

    report = numeric_format_report.get(raw_col)

    # =====================================================
    # KASUS 1:
    # Ditemukan nilai bertitik-ganda (mis. 2.226.401) DAN pola
    # TERBUKTI konsisten (seluruh grup setelah grup pertama = 3
    # digit, di SELURUH nilai bertitik-ganda pada kolom ini)
    # -> AMAN diparsing sebagai pemisah ribuan.
    # =====================================================
    if report and report["n_multi_dot_values"] > 0 and report["consistent_thousand_sep"]:

        sounderpy_raw[raw_col + "_raw_string"] = sounderpy_raw[raw_col].astype(str)
        sounderpy_raw[raw_col] = sounderpy_raw[raw_col].apply(parse_thousand_separated)
        columns_parsed.append(raw_col)

        print(
            f"[DIPARSING] {raw_col}: {report['n_multi_dot_values']} nilai bertitik-ganda, "
            f"pola pemisah-ribuan terbukti konsisten (0 pelanggaran) -> dikonversi ke numerik."
        )

    # =====================================================
    # KASUS 2:
    # Ditemukan nilai bertitik-ganda TAPI pola TIDAK terbukti
    # konsisten (ada pelanggaran) -> JANGAN diparsing, simpan
    # apa adanya, tandai sebagai temuan terbuka.
    # =====================================================
    elif report and report["n_multi_dot_values"] > 0 and not report["consistent_thousand_sep"]:

        sounderpy_raw[raw_col + "_raw_string"] = sounderpy_raw[raw_col].astype(str)
        columns_flagged_unresolved.append(raw_col)

        print(
            f"[TEMUAN TERBUKA] {raw_col}: {report['n_multi_dot_values']} nilai bertitik-ganda, "
            f"{report['n_violations']} di antaranya melanggar pola pemisah-ribuan "
            f"(contoh pelanggaran: {report['example_violations']}). Pola TIDAK terbukti "
            f"konsisten -> nilai dipertahankan apa adanya (string), tidak diparsing secara spekulatif."
        )

    # =====================================================
    # KASUS 3:
    # Tidak ada nilai bertitik-ganda sama sekali -> konversi numerik normal
    # =====================================================
    else:
        converted = pd.to_numeric(sounderpy_raw[raw_col], errors="coerce")
        n_failed = converted.isna().sum() - sounderpy_raw[raw_col].isna().sum()
        if n_failed > 0:
            print(f"[PERINGATAN] {raw_col}: {n_failed} nilai tidak dapat dikonversi ke numerik")
        sounderpy_raw[raw_col] = converted

print("\n=== RINGKASAN TEMUAN ===")
print("Kolom yang DIPARSING (pola pemisah-ribuan terbukti konsisten):")
print(columns_parsed if columns_parsed else "Tidak ada")
print("\nKolom dengan nilai bertitik-ganda yang TIDAK terbukti konsisten (memerlukan tinjauan lanjutan):")
print(columns_flagged_unresolved if columns_flagged_unresolved else "Tidak ada")


> **Catatan penting:** kolom `..._raw_string` di atas HANYA untuk audit/dokumentasi, bukan
> bagian dari output final Tahap 2 dan tidak boleh dianggap sebagai fitur. Jika ada kolom yang
> masuk kategori "temuan terbuka", nilai yang tidak terkonversi menjadi `NaN` pada kolom numerik
> — ini BUKAN imputasi (Tahap 7), melainkan efek samping dari `pd.to_numeric(errors="coerce")`
> pada string yang memang tidak bisa diinterpretasikan secara aman. Jumlahnya dilaporkan di
> sel ringkasan (bagian 12) agar tidak hilang tanpa jejak.


## 11. SounderPy — Harmonisasi Nama Kolom & Susun Output

Rename ke konvensi final dan pilih kolom akhir:
`date`, `hour`, `cape`, `cin`, `kindex`, `li`, `tt`, `sweat`.


In [ ]:
rename_map = {raw: final for raw, final in RAW_VAR_COLS_SOUNDERPY.items() if raw in sounderpy_raw.columns}
sounderpy_standardized = sounderpy_raw.rename(columns=rename_map)

for raw_col, final_name in RAW_VAR_COLS_SOUNDERPY.items():
    if raw_col in columns_flagged_unresolved:
        sounderpy_standardized[final_name] = (
            sounderpy_standardized[raw_col + "_raw_string"]
        )

# Pertahankan metadata mentah agar investigasi pada Tahap berikutnya tetap memungkinkan
# (bukan kolom baru — hanya TIDAK dibuang dari data yang sudah ada).
METADATA_COLS_TO_KEEP = [c for c in ["nominal_date", "observation_datetime"] if c in sounderpy_standardized.columns]
_missing_metadata = [c for c in ["nominal_date", "observation_datetime"] if c not in sounderpy_standardized.columns]
if _missing_metadata:
    print(f"[CATATAN] Kolom metadata berikut tidak ditemukan di data mentah, tidak bisa dipertahankan: {_missing_metadata}")

final_cols = ["date", "hour"] + METADATA_COLS_TO_KEEP + list(rename_map.values())
sounderpy_standardized = sounderpy_standardized[final_cols].copy()

print("Tipe data akhir:")
print(sounderpy_standardized.dtypes)
sounderpy_standardized.head()


## 12. SounderPy — Ringkasan Temuan Numerik (Untuk Dibawa ke Tahap Berikutnya)

In [ ]:
summary_rows = []

for raw_col, final_name in RAW_VAR_COLS_SOUNDERPY.items():

    if raw_col not in numeric_format_report:
        continue

    r = numeric_format_report[raw_col]

    n_nan_after = (
        sounderpy_standardized[final_name].isna().sum()
        if final_name in sounderpy_standardized.columns
        else None
    )

    if raw_col in columns_flagged_unresolved:
        status = "TEMUAN_TERBUKA"
    elif raw_col in columns_parsed:
        status = "DIPARSING"
    else:
        status = "BERSIH"

    summary_rows.append({
        "kolom_final": final_name,
        "kolom_mentah": raw_col,
        "n_nilai_bertitik_ganda": r["n_multi_dot_values"],
        "pola_terbukti_konsisten": r["consistent_thousand_sep"],
        "status": status,
        "contoh_nilai": r["example_matches"],
        "n_nan_setelah_standardisasi": n_nan_after,
    })

numeric_findings_summary = pd.DataFrame(summary_rows)

numeric_findings_summary


## 13. SounderPy — Simpan Output

Sesuai instruksi: duplicate tidak relevan di sini (tidak ada instruksi drop untuk SounderPy),
jumlah baris tidak boleh berubah dari hasil load mentah.


In [ ]:
assert len(sounderpy_standardized) == len(sounderpy_raw), (
    "Jumlah baris SounderPy berubah selama standardisasi — seharusnya tidak ada baris "
    "yang ditambah/dihapus di Tahap 2."
)

sounderpy_export = sounderpy_standardized.copy()
sounderpy_export["date"] = sounderpy_export["date"].dt.strftime("%Y-%m-%d")

sounderpy_out_path = os.path.join(OUTPUT_DIR, "sounderpy_standardized.csv")
sounderpy_export.to_csv(sounderpy_out_path, index=False)
print(f"Tersimpan: {sounderpy_out_path}")


---
## 14. Checkpoint Tahap 2

Jalankan sel ini sebagai verifikasi akhir sebelum lanjut ke Tahap 3 (Validasi Kalender).


In [ ]:
print("=== CHECKPOINT TAHAP 2 ===\n")

print("[Ogimet]")
print(f"- Baris   : {len(ogimet_standardized)}")
print(f"- Kolom   : {list(ogimet_standardized.columns)}")
print(f"- Tanggal NaT: {ogimet_standardized['date'].isna().sum()}")
print(f"- Tanggal min/max: {ogimet_standardized['date'].min()} s/d {ogimet_standardized['date'].max()}")
print(f"- Tipe numerik OK: {all(pd.api.types.is_numeric_dtype(ogimet_standardized[c]) for c in ['rr','tavg','rh'])}")
print(f"- Tipe date == datetime64: {pd.api.types.is_datetime64_any_dtype(ogimet_standardized['date'])}")
print(f"- Duplicate rows dipertahankan: {ogimet_standardized.duplicated().sum()}")

print("\n[SounderPy]")
print(f"- Baris   : {len(sounderpy_standardized)}")
print(f"- Kolom   : {list(sounderpy_standardized.columns)}")
print(f"- Tanggal NaT: {sounderpy_standardized['date'].isna().sum()}")
print(f"- Tanggal min/max: {sounderpy_standardized['date'].min()} s/d {sounderpy_standardized['date'].max()}")
print(f"- Tipe date == datetime64: {pd.api.types.is_datetime64_any_dtype(sounderpy_standardized['date'])}")
print(f"- Duplicate (date+hour): {sounderpy_standardized.duplicated(subset=['date','hour']).sum()}")
print(f"- Kolom DIPARSING dari format pemisah-ribuan: {columns_parsed}")
print(f"- Kolom dengan temuan format numerik terbuka: {columns_flagged_unresolved}")

print("\n[Output tersimpan]")
print(f"- {ogimet_out_path}")
print(f"- {sounderpy_out_path}")

print("\n[Risiko tercatat, TIDAK diselesaikan di Tahap 2]")
print("1. Interpretasi UTC (00Z/12Z) vs kalender lokal WIB — dibawa ke Tahap 3/4/5.")
if columns_flagged_unresolved:
    print(f"2. Format numerik belum terparsing (pola tak terbukti konsisten) di kolom: {columns_flagged_unresolved} — perlu tinjauan peneliti sebelum Tahap 5/6.")


## 15. Yang Sengaja TIDAK Dilakukan di Notebook Ini

Sesuai batasan Tahap 2:
- ❌ Drop duplicate
- ❌ Drop missing value
- ❌ Imputasi
- ❌ Interpolasi
- ❌ Merge dataset (Ogimet + SounderPy)
- ❌ Seleksi sounding 12Z / 00Z
- ❌ Master calendar
- ❌ Labeling
- ❌ Feature engineering

Seluruh poin di atas ditangani pada Tahap 3–11 sesuai playbook.


In [ ]:
print("OGIMET")
print("Duplicate date:", ogimet_standardized["date"].duplicated().sum())
print("Missing date:", ogimet_standardized["date"].isna().sum())
print("Tanggal minimum :", ogimet_standardized["date"].min())
print("Tanggal maksimum:", ogimet_standardized["date"].max())


In [20]:
ogimet_standardized[
    ogimet_standardized["date"].duplicated(keep=False)
].sort_values("date").head(30)

,date,rr,tavg,rh
2922,NaN,NaN,NaN,NaN
2923,NaN,NaN,NaN,NaN
2924,NaN,NaN,NaN,NaN
2925,NaN,NaN,NaN,NaN
2926,NaN,NaN,NaN,NaN
2927,NaN,NaN,NaN,NaN
2928,NaN,NaN,NaN,NaN
2929,NaN,NaN,NaN,NaN
2930,NaN,NaN,NaN,NaN
2931,NaN,NaN,NaN,NaN


In [ ]:
print("SOUNDERPY")
print("Duplicate (date+hour):", sounderpy_standardized.duplicated(subset=["date", "hour"]).sum())
print("Missing date:", sounderpy_standardized["date"].isna().sum())
print("Tanggal minimum :", sounderpy_standardized["date"].min())
print("Tanggal maksimum:", sounderpy_standardized["date"].max())


In [ ]:
sounderpy_standardized[
    sounderpy_standardized.duplicated(subset=["date", "hour"], keep=False)
].sort_values(["date", "hour"]).head(50)
